In [1]:
!pip install -U langchain
!pip install -U langchain-google-genai
!pip install -U google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 1.5 MB/s eta 0:00:00


In [2]:
import os

os.environ["Gemini Api Key"] = "AQ.Ab8RN6L1JHXGrebf9LMbWNEP9fnhTwzyVH16XgKNXwr8mauuWw"

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=os.environ["Gemini Api Key"]
)

respuesta = llm.invoke("Hola, preséntate en una sola oración.")
print(respuesta.content)

Soy un modelo de lenguaje grande, entrenado por Google.


In [4]:
from langchain_core.prompts import PromptTemplate

template = """
Eres un vendedor de ropa para niños.

Responde únicamente usando la información del inventario.

Inventario:
{inventario}

Pregunta:
{pregunta}
"""

prompt = PromptTemplate(
    input_variables=["inventario", "pregunta"],
    template=template
)

In [5]:
import pandas as pd

data = {
    'Producto': ['Polo niño avenger', 'Polo niño spiderman', 'Conjunto niño deportivo', 'Casaca niño invierno', 'Pantalón niño jean'],
    'Categoría': ['Ropa niño', 'Ropa niño', 'Ropa niño', 'Ropa niño', 'Ropa niño'],
    'Talla': [4, 6, 8, 10, 12],
    'Precio': [35, 38, 65, 75, 50],
    'Stock': [20, 15, 10, 8, 12]
}
df = pd.DataFrame(data)

inventario = df.to_string(index=False)

print(inventario)

               Producto Categoría  Talla  Precio  Stock
      Polo niño avenger Ropa niño      4      35     20
    Polo niño spiderman Ropa niño      6      38     15
Conjunto niño deportivo Ropa niño      8      65     10
   Casaca niño invierno Ropa niño     10      75      8
     Pantalón niño jean Ropa niño     12      50     12


In [6]:
mensaje = prompt.format(
    inventario=inventario,
    pregunta="¿Qué productos tienes disponibles?"
)

respuesta = llm.invoke(mensaje)

print(respuesta.content)

Tenemos disponibles los siguientes productos:

*   Polo niño avenger
*   Polo niño spiderman
*   Conjunto niño deportivo
*   Casaca niño invierno
*   Pantalón niño jean


In [7]:
pregunta = "¿cual es el polo de niño que tiene mas stock?"

mensaje = prompt.format(
    inventario=inventario,
    pregunta=pregunta
)

respuesta = llm.invoke(mensaje)

print(respuesta.content)

El polo de niño que tiene más stock es el Polo niño avenger con 20 unidades.


In [8]:
from langchain.tools import tool

@tool
def consultar_inventario():
    """Devuelve el inventario de productos."""
    return df.to_string(index=False)

In [9]:
print(consultar_inventario.invoke({}))

               Producto Categoría  Talla  Precio  Stock
      Polo niño avenger Ropa niño      4      35     20
    Polo niño spiderman Ropa niño      6      38     15
Conjunto niño deportivo Ropa niño      8      65     10
   Casaca niño invierno Ropa niño     10      75      8
     Pantalón niño jean Ropa niño     12      50     12


In [10]:
import langgraph

In [11]:
from langgraph.prebuilt import create_react_agent

In [12]:
tools = [consultar_inventario]

print(tools)

[StructuredTool(name='consultar_inventario', description='Devuelve el inventario de productos.', args_schema=<class 'langchain_core.utils.pydantic.consultar_inventario'>, func=<function consultar_inventario at 0x7d7ceee51580>)]


In [13]:
print(type(llm))

<class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>


In [14]:
agent = create_react_agent(
    model=llm,
    tools=tools
)

/tmp/ipykernel_490/1123741138.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [15]:
respuesta = agent.invoke(
    {
        "messages": [
            ("human", "¿Qué productos tienes disponibles?")
        ]
    }
)

In [16]:
type(respuesta)

dict

In [17]:
respuesta

{'messages': [HumanMessage(content='¿Qué productos tienes disponibles?', additional_kwargs={}, response_metadata={}, id='2fcfb2e2-10fd-4fa8-b7a6-8d00d8eb989f'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'consultar_inventario', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'16c57582-8946-4cd4-8993-26f72e0a912a': 'CvUBARFNMg/mIRpTyP8BbMwEkKu9vy0aigkNFLAWWRxGt2ySYtXdAKEu23DDkHH0PiHnNEw7FxZ2na8UzNDNNJ3I6OueI3exq7IkJJJS05AqPUJ62U7FtklPN0Sa0e4IEVJHLH948/ClX6JCQchIUSeW7Mx8GxjChu7yVsu9JopQIQXsP7rCAJjoELHOF9Q0sh0eiCw3Llutyuz069n5a3hPX82gy+lv8JowtxsV9KrMlQYwnaHuJcNX/XJxG1uwBo9d31advzx3v3WVMvetzXi1gKK0QpyLOoQS8HAd64AtdI7TPMe9ES0fz3VaQzOGdt7zrsJduxA='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f7830-dcb5-7532-b638-5a59dde4f153-0', tool_calls=[{'name': 'consultar_inventario', 'args': {}, 'id': '16c57582-8946-4cd4-8993-26f72e0a912a', 'type': '

In [19]:
respuesta = agent.invoke(
    {"messages": [("user", "¿Qué productos tienes?")]}
)

print(type(respuesta))
print(respuesta.keys())

<class 'dict'>
dict_keys(['messages'])


In [20]:
print(respuesta["messages"][-1])

content='Tenemos los siguientes productos en inventario:\n\n*   **Polo niño avenger**: Ropa niño, talla 4, precio 35, stock 20\n*   **Polo niño spiderman**: Ropa niño, talla 6, precio 38, stock 15\n*   **Conjunto niño deportivo**: Ropa niño, talla 8, precio 65, stock 10\n*   **Casaca niño invierno**: Ropa niño, talla 10, precio 75, stock 8\n*   **Pantalón niño jean**: Ropa niño, talla 12, precio 50, stock 12' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f7833-acfe-7a51-95db-320efa4219d5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 168, 'output_tokens': 142, 'total_tokens': 310, 'input_token_details': {'cache_read': 0}}


In [21]:
print(respuesta["messages"][-1].content)

Tenemos los siguientes productos en inventario:

*   **Polo niño avenger**: Ropa niño, talla 4, precio 35, stock 20
*   **Polo niño spiderman**: Ropa niño, talla 6, precio 38, stock 15
*   **Conjunto niño deportivo**: Ropa niño, talla 8, precio 65, stock 10
*   **Casaca niño invierno**: Ropa niño, talla 10, precio 75, stock 8
*   **Pantalón niño jean**: Ropa niño, talla 12, precio 50, stock 12


In [22]:
def preguntar(pregunta):
    respuesta = agent.invoke(
        {"messages": [("user", pregunta)]}
    )
    return respuesta["messages"][-1].content

In [ ]:
# ==========================
# PRUEBAS DEL AGENTE
# ==========================

In [23]:
print(preguntar("¿Qué productos tienes?"))

print(preguntar("¿Cuál es el producto más barato?"))

print(preguntar("¿Qué polo tiene mayor stock?"))

print(preguntar("¿Cuánto cuesta la casaca niño invierno?"))

Actualmente, tengo los siguientes productos en inventario:

*   **Polo niño avenger**: Categoría Ropa niño, talla 4, precio 35, stock 20.
*   **Polo niño spiderman**: Categoría Ropa niño, talla 6, precio 38, stock 15.
*   **Conjunto niño deportivo**: Categoría Ropa niño, talla 8, precio 65, stock 10.
*   **Casaca niño invierno**: Categoría Ropa niño, talla 10, precio 75, stock 8.
*   **Pantalón niño jean**: Categoría Ropa niño, talla 12, precio 50, stock 12.
El producto más barato es el Polo niño avenger, con un precio de 35.
El polo de niño avenger tiene 20 unidades de stock, siendo el de mayor cantidad.
[{'type': 'text', 'text': 'La casaca niño invierno cuesta 75 soles.', 'extras': {'signature': 'CrQDARFNMg9vSS4wiZotLhe34xxcDQzLy7BuXm5cFwxX/thoIovkjmNawF2JAriHtaCsIoeaed2ESWXqytpzmM3mZVjuVyhZjNgTNF7Ow+bMtF3gDjLWkQmnWEhN5eBos30M084xqDujIMdgXRXE1ZGJ54pFGpjY0aK3mWm3aOAHRtLz9MIQpYzJFI+ozPcCUKiVMWyQ3eK6KT+UQvsUiJJo42C9OLefKcF2ALEXX+YNeqbNNMrkszW7sMn9O35MIpykuSOHdXzA1r20SWzdfPFK1Fsc03gJoGP1